In [14]:
import pandas as pd
import numpy as np
import os
import re

In [15]:
# Read data
# data = "../data/skincare_mixing_dummy.xlsx"
file_path  = "../data/sample_batch_83012_from_screenshot.xlsx"

### Step 3 — Read the Excel without assuming a header

In [16]:
df_raw = pd.read_excel(
    file_path,
    sheet_name="Table 1",
    header=None
)

In [19]:
print(df_raw.head(20))

           0          1               2              3         4   \
0         NaN        NaN             NaN            NaN       NaN   
1         NaN        NaN             NaN            NaN       NaN   
2         NaN        NaN             NaN            NaN       NaN   
3         NaN        NaN             NaN            NaN       NaN   
4         NaN        NaN             NaN            NaN       NaN   
5         NaN        NaN             NaN            NaN       NaN   
6         NaN        NaN             NaN            NaN       NaN   
7   Unique ID  Unit Name  Unit Procedure      Operation     Phase   
8       83012   MIXER_M5             NaN            NaN       NaN   
9         NaN        NaN             NaN  OP_AQ2_M5_JBL       NaN   
10        NaN        NaN             NaN            NaN  PROMPT:1   
11        NaN        NaN             NaN            NaN       NaN   
12        NaN        NaN             NaN            NaN       NaN   
13        NaN        NaN          

### Step 4 — Find the actual table header

In [20]:
required_columns = [
    "Unique ID",
    "Unit Name",
    "Start Time",
    "End Time",
    "Duration"
]

In [21]:
# Now search each row for the required columns and extract the data
header_row = None

for i in range(len(df_raw)):

    row_values = df_raw.iloc[i].astype(str).str.strip().tolist()

    if all(col in row_values for col in required_columns):
        header_row = i
        break

print("Header row:", header_row)

Header row: 7


### Step 5 — Extract the main table

In [22]:
headers = df_raw.iloc[header_row].tolist()

print(headers)

['Unique ID', 'Unit Name', 'Unit Procedure', 'Operation', 'Phase', 'Start Time', 'End Time', 'Duration', nan, nan, nan, np.float64(nan), nan, np.float64(nan), nan]


dataset 1 (df_timestamp)- strt_time, end_time, duration

In [23]:
# create a dataframe 
df = df_raw.iloc[header_row + 1:].copy()

df.columns = headers

In [24]:
print(df.head())

   Unique ID Unit Name Unit Procedure      Operation     Phase  \
8      83012  MIXER_M5            NaN            NaN       NaN   
9        NaN       NaN            NaN  OP_AQ2_M5_JBL       NaN   
10       NaN       NaN            NaN            NaN  PROMPT:1   
11       NaN       NaN            NaN            NaN       NaN   
12       NaN       NaN            NaN            NaN       NaN   

             Start Time             End Time  Duration               NaN  \
8   2026-01-13 13:05:18  2026-01-13 20:46:23   7:41:05               NaN   
9   2026-01-13 13:05:18  2026-01-13 20:46:23  07:41:05               NaN   
10  2026-01-13 13:05:18  2026-01-13 13:05:30  00:00:12               NaN   
11                  NaN                  NaN       NaN  Recipe Parameter   
12                  NaN                  NaN       NaN           MESSAGE   

                      NaN  NaN  NaN  NaN  NaN  NaN  
8                     NaN  NaN  NaN  NaN  NaN  NaN  
9                     NaN  NaN  NaN  NaN

### Step 6 — Keep only the relevant columns

In [25]:
required_columns = [
    "Unique ID",
    "Unit Name",
    "Unit Procedure",
    "Operation",
    "Phase",
    "Start Time",
    "End Time",
    "Duration"
]

df = df[required_columns].copy()

In [26]:
df.head(10)

,Unique ID,Unit Name,Unit Procedure,Operation,Phase,Start Time,End Time,Duration
8,83012,MIXER_M5,NaN,NaN,NaN,2026-01-13 13:05:18,2026-01-13 20:46:23,7:41:05
9,NaN,NaN,NaN,OP_AQ2_M5_JBL,NaN,2026-01-13 13:05:18,2026-01-13 20:46:23,07:41:05
10,NaN,NaN,NaN,NaN,PROMPT:1,2026-01-13 13:05:18,2026-01-13 13:05:30,00:00:12
11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,NaN,NaN,NaN,NaN,PROMPT:7,2026-01-13 13:05:30,2026-01-13 13:05:35,00:00:05
17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 7 — Remove completely empty rows

In [27]:
df = df.dropna(
    how="all"
).reset_index(drop=True)

In [28]:
df

,Unique ID,Unit Name,Unit Procedure,Operation,Phase,Start Time,End Time,Duration
0,83012,MIXER_M5,NaN,NaN,NaN,2026-01-13 13:05:18,2026-01-13 20:46:23,7:41:05
1,NaN,NaN,NaN,OP_AQ2_M5_JBL,NaN,2026-01-13 13:05:18,2026-01-13 20:46:23,07:41:05
2,NaN,NaN,NaN,NaN,PROMPT:1,2026-01-13 13:05:18,2026-01-13 13:05:30,00:00:12
3,NaN,NaN,NaN,NaN,PROMPT:7,2026-01-13 13:05:30,2026-01-13 13:05:35,00:00:05
4,NaN,NaN,NaN,NaN,PROMPT:12,2026-01-13 13:05:36,2026-01-13 13:05:44,00:00:08


### Step 8 — Understand the hierarchy

### Step 9 — Forward-fill the hierarchy

In [29]:
hierarchy_columns = [
    "Unique ID",
    "Unit Name",
    "Unit Procedure",
    "Operation"
]

df[hierarchy_columns] = (
    df[hierarchy_columns]
    .ffill()
)

In [30]:
df

,Unique ID,Unit Name,Unit Procedure,Operation,Phase,Start Time,End Time,Duration
0,83012,MIXER_M5,NaN,NaN,NaN,2026-01-13 13:05:18,2026-01-13 20:46:23,7:41:05
1,83012,MIXER_M5,NaN,OP_AQ2_M5_JBL,NaN,2026-01-13 13:05:18,2026-01-13 20:46:23,07:41:05
2,83012,MIXER_M5,NaN,OP_AQ2_M5_JBL,PROMPT:1,2026-01-13 13:05:18,2026-01-13 13:05:30,00:00:12
3,83012,MIXER_M5,NaN,OP_AQ2_M5_JBL,PROMPT:7,2026-01-13 13:05:30,2026-01-13 13:05:35,00:00:05
4,83012,MIXER_M5,NaN,OP_AQ2_M5_JBL,PROMPT:12,2026-01-13 13:05:36,2026-01-13 13:05:44,00:00:08


### Step 10 — Convert Unique ID to string

In [31]:
df["Unique ID"] = (
    df["Unique ID"]
    .astype("Int64")
    .astype(str)
)

In [32]:
df["Unique ID"] = df["Unique ID"].astype("string")
print(df["Unique ID"])

0    83012
1    83012
2    83012
3    83012
4    83012
Name: Unique ID, dtype: string


### Step 13 — Convert Start Time

In [ ]:
df["Start Time"] = pd.to_datetime(
    df["Start Time"],
    errors="coerce"
)


In [34]:
df["End Time"] = pd.to_datetime(
    df["End Time"],
    errors="coerce"
)

In [35]:
print(
    df[
        ["Start Time", "End Time"]
    ].dtypes
)

Start Time    datetime64[us]
End Time      datetime64[us]
dtype: object
